# 第 2 周练习 —— 国家探索（Country Explorer）

## 练习目标（理念）

做一个**多模态**国家问答机器人：真实国家数据 + SVG 地图轮廓 + 文本转语音（TTS）。

## 技能对照

| 技能 | 本练习落点 |
|------|------------|
| Gradio Blocks | 聊天 + 地图 + 音频自定义布局 |
| 流式展示 | `respond` 按字符 `yield` |
| Tool / Function Calling | `get_country_data` / `generate_country_map` |
| 多模型切换 | GPT / Claude / Gemini |
| TTS | `openai_client.audio.speech` |

## 怎么跑

1. `.env`：OpenAI 密钥 + `OPENROUTER_API_KEY`（Claude/Gemini）
2. 从上到下运行；可先单独测两个工具函数
3. 最后 `ui.launch(inbrowser=True)` 打开界面


In [ ]:
# ========== 导入：HTTP / OpenAI / Gradio / 临时文件 ==========

# 导入标准库 os：读环境变量
import os
# 导入标准库 json：解析 tool call 的 arguments
import json
# 导入 tempfile：TTS 音频落盘用临时 mp3 路径
import tempfile
# 导入 requests：调用 REST Countries 与 GitHub 上的 SVG
import requests
# 从 dotenv 导入 load_dotenv：加载 .env
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：Chat Completions + TTS
from openai import OpenAI
# 从 IPython.display 导入：笔记本里预览 SVG（HTML）
from IPython.display import display, HTML
# 导入 gradio：Blocks 自定义布局
import gradio as gr


In [ ]:
# ========== 初始化：环境变量 + 双客户端 + 默认模型名 ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)

# 官方 OpenAI 客户端（GPT + TTS）
openai_client = OpenAI()
# OpenRouter：用来打 Claude / Gemini（URL 与环境变量名保持原样）
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

# 默认模型常量（本练习主路径仍多用 model_config）
MODEL = "gpt-4.1-mini"


In [ ]:
# ========== 模型切换配置：展示名 → (client, model_id) ==========
# Dropdown 的选项就是这里的 key

model_config = {
    "GPT": (openai_client, "gpt-4.1-mini"),
    "Claude": (openrouter_client, "anthropic/claude-3.5-haiku"),
    "Gemini": (openrouter_client, "google/gemini-2.5-flash-lite"),
}


---
## 工具（Tools）

LLM 可调用两种工具：一种拉**真实国家数据**，一种取 **SVG 地图轮廓**。


In [ ]:
# ========== 工具 1：REST Countries 拉国家事实 ==========
# 返回给模型的多行英文摘要字符串保持原样（勿改字段名/文案）

# REST Countries 按名称查询的基础 URL
API_URL = "https://restcountries.com/v3.1/name"
# fields 查询参数：只取需要的字段，减小响应
API_FIELDS = "name,capital,population,languages,currencies,region,subregion,flag,timezones,cca2"

def get_country_data(country_name):
    try:
        # GET /name/{country_name}?fields=...
        resp = requests.get(
            f"{API_URL}/{country_name}",
            params={"fields": API_FIELDS},
            timeout=10
        )
        # 非 200：多半是找不到国家
        if resp.status_code != 200:
            return f"Could not find a country called '{country_name}'."
        # API 返回列表，取第一条
        data = resp.json()[0]
        # languages / currencies 是字典，拼成可读字符串
        languages = ", ".join(data.get("languages", {}).values())
        currencies = ", ".join(
            f"{v['name']} ({v['symbol']})" for v in data.get("currencies", {}).values()
        )
        # 拼多行摘要（标签英文保持原样）
        return (
            f"{data.get('flag', '')} {data['name']['official']}\n"
            f"Capital: {data.get('capital', ['Unknown'])[0]}\n"
            f"Population: {data['population']:,}\n"
            f"Languages: {languages}\n"
            f"Currencies: {currencies}\n"
            f"Region: {data.get('region', '')} -- {data.get('subregion', '')}\n"
            f"Timezones: {', '.join(data.get('timezones', []))}"
        )
    except requests.RequestException as e:
        return f"Network error: {e}"


In [ ]:
# ========== 试跑工具 1：真实国家 Rwanda ==========

get_country_data("Rwanda")


In [ ]:
# ========== 试跑工具 1：虚构国家 Wakanda（应失败提示） ==========

get_country_data("Wakanda")


In [ ]:
# ========== 工具 2：mapsicon SVG 轮廓 ==========
# 先查 cca2 国家码，再拼 GitHub raw SVG URL（URL 模板保持原样）

MAP_SVG_URL = "https://raw.githubusercontent.com/djaiss/mapsicon/master/all/{cca2}/vector.svg"
CCA2_API = "https://restcountries.com/v3.1/name/{name}?fields=cca2"

def generate_country_map(country_name):
    try:
        # 第一步：国家名 → ISO 3166-1 alpha-2（小写）
        resp = requests.get(CCA2_API.format(name=country_name), timeout=10)
        if resp.status_code != 200:
            return None
        cca2 = resp.json()[0]["cca2"].lower()
        # 第二步：下载 SVG 文本
        svg_resp = requests.get(MAP_SVG_URL.format(cca2=cca2), timeout=10)
        if svg_resp.status_code != 200:
            return None
        return svg_resp.text
    except (requests.RequestException, KeyError, IndexError):
        # 网络错误或 JSON 缺字段 → 无地图
        return None


In [ ]:
# ========== 试跑工具 2：在笔记本里预览 Burundi 地图 ==========

svg = generate_country_map("Burundi")
if svg:
    # 把 SVG 当 HTML 嵌入显示
    display(HTML(svg))
else:
    # 失败提示文案保持原样
    print("Could not fetch map")


---
## 工具架构（Tool Schema）

下面的 JSON 定义告诉 LLM「有哪些工具、参数是什么」。


In [ ]:
# ========== Function schema：get_country_data ==========
# description / parameters 英文保持原样（模型靠它们决定是否调用）

country_data_function = {
    "name": "get_country_data",
    "description": "Get verified country info: capital, population, languages, currencies, region",
    "parameters": {
        "type": "object",
        "properties": {
            "country_name": {
                "type": "string",
                "description": "The country name"
            }
        },
        "required": ["country_name"],
        "additionalProperties": False
    }
}


In [ ]:
# ========== Function schema：generate_country_map ==========

country_map_function = {
    "name": "generate_country_map",
    "description": "Fetch an SVG map showing a country's geographic outline",
    "parameters": {
        "type": "object",
        "properties": {
            "country_name": {
                "type": "string",
                "description": "The country name"
            }
        },
        "required": ["country_name"],
        "additionalProperties": False
    }
}


In [ ]:
# ========== 组装 tools 列表：供 chat.completions.create(tools=...) ==========

tools = [
    {"type": "function", "function": country_data_function},
    {"type": "function", "function": country_map_function},
]


---
## 聊天引擎（Chat Engine）

系统提示、工具处理函数、以及「请求 → 工具 → 再请求」的代理循环。


In [ ]:
# ========== System message：国家专家人设（英文正文保持原样） ==========
# 关键点：地图在侧栏展示，要求模型回复里不要提「正在生成地图」

system_message = """You are a knowledgeable and friendly country expert.
When asked about a country, use get_country_data for verified facts
and generate_country_map for a visual map outline.
The map is displayed in a separate panel -- do not mention or reference map generation in your response.
Present the data engagingly and include a fun fact the user might not know.
If the question isn't about a country, respond normally but mention your specialty."""


In [ ]:
# ========== handle_tool_calls：执行模型请求的工具 ==========
# 返回 (tool messages 列表, svg 或 None)；地图工具把 SVG 留在本地变量给 UI


def handle_tool_calls(message):
    responses = []
    svg = None
    # 可能一次请求多个 tool_calls
    for tc in message.tool_calls:
        # arguments 是 JSON 字符串 → dict
        args = json.loads(tc.function.arguments)
        name = tc.function.name
        if name == "get_country_data":
            result = get_country_data(args["country_name"])
        elif name == "generate_country_map":
            svg = generate_country_map(args["country_name"])
            # 回给模型的短状态句保持原样（真正 SVG 走侧栏）
            result = "Map displayed to the user." if svg else "Map unavailable."
        else:
            result = "Unknown tool"
        # role=tool 必须带 tool_call_id 对齐
        responses.append({"role": "tool", "content": result, "tool_call_id": tc.id})
    return responses, svg


In [ ]:
# ========== chat：代理工具循环，返回 (最终文本, svg) ==========


def chat(history, model_name):
    # 按 Dropdown 选中的名字取出 client 与 model id
    client, model = model_config[model_name]
    # 以 system 开头，再拼上 Gradio messages 历史
    messages = [{"role": "system", "content": system_message}]
    messages += [{"role": h["role"], "content": h["content"]} for h in history]

    svg_result = None

    # 代理工具调用循环：先发带 tools 的请求
    response = client.chat.completions.create(model=model, messages=messages, tools=tools)

    # finish_reason == tool_calls 时：执行工具 → 把结果塞回 messages → 再请求
    while response.choices[0].finish_reason == "tool_calls":
        tool_msg = response.choices[0].message
        tool_responses, svg = handle_tool_calls(tool_msg)
        if svg:
            svg_result = svg
        # 助手的 tool_calls 消息本身也要进历史
        messages.append(tool_msg)
        messages.extend(tool_responses)
        response = client.chat.completions.create(model=model, messages=messages, tools=tools)

    # 最终自然语言回复；可能为空则用 ""
    reply = response.choices[0].message.content or ""
    return reply, svg_result


---
## 多模态：文本转语音（TTS）


In [ ]:
# ========== talker：OpenAI TTS → 临时 mp3 路径 ==========
# 失败时返回 None（UI 侧可忽略音频）


def talker(text):
    try:
        # gpt-4o-mini-tts + voice=onyx；只朗读前 500 字符
        response = openai_client.audio.speech.create(
            model="gpt-4o-mini-tts",
            voice="onyx",
            input=text[:500]
        )
        # 写入临时 mp3，delete=False 以便 Gradio 还能读到文件
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
            f.write(response.content)
            return f.name
    except Exception:
        return None


---
## Gradio 用户界面（Blocks）

用 `gr.Blocks` 做自定义布局：聊天面板 + SVG 地图 + 音频 + 模型切换器。


In [ ]:
# ========== Gradio 回调：先塞用户消息，再流式回复 + TTS ==========


def put_message_in_chatbot(message, history):
    # 空输入：原样返回，不改 history
    if not message.strip():
        return message, history
    # 清空输入框，并把 user 消息追加进 chatbot
    return "", history + [{"role": "user", "content": message}]


def respond(history, model_name):
    # 防御：没有以 user 结尾的 history 则空 yield
    if not history or history[-1]["role"] != "user":
        yield history, None, None
        return

    # 调用代理聊天：拿到完整回复与可选 SVG
    reply, svg_result = chat(history, model_name)

    # 把 SVG 包进居中容器（HTML 样式字符串保持原样）
    svg_html = (
        f'<div style="display:flex;justify-content:center;align-items:center;'
        f'height:100%;max-width:350px;max-height:350px;margin:auto">{svg_result}</div>'
        if svg_result else None
    )

    # 将回复流式传输至聊天机器人（按字符假流式，便于 UI 动起来）
    streamed = ""
    for char in reply:
        streamed += char
        yield history + [{"role": "assistant", "content": streamed}], svg_html, None

    # 文本完成后生成音频，最后一次 yield 带上 audio 路径
    audio = talker(reply)
    yield history + [{"role": "assistant", "content": reply}], svg_html, audio


In [ ]:
# ========== Blocks UI 定义与事件链 ==========

# title 仅影响浏览器标签；页面标题用 Markdown
with gr.Blocks(title="Country Explorer") as ui:
    gr.Markdown("### Country Explorer")

    # 左聊天、右地图
    with gr.Row():
        chatbot = gr.Chatbot(height=400, type="messages")
        map_display = gr.HTML(
            value='<div style="height:350px;display:flex;align-items:center;'
                  'justify-content:center;color:#888">Map appears here</div>'
        )

    # 音频自动播放
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)

    # 输入框 + 模型下拉
    with gr.Row():
        message = gr.Textbox(label="Ask about a country:", placeholder="Tell me about Japan...", scale=3)
        model_dropdown = gr.Dropdown(list(model_config.keys()), value="GPT", label="Model", scale=1)

    # submit：先 put_message_in_chatbot，再 then(respond) 流式更新三路输出
    message.submit(
        put_message_in_chatbot, [message, chatbot], [message, chatbot]
    ).then(
        respond, [chatbot, model_dropdown], [chatbot, map_display, audio_output]
    )

# inbrowser=True：启动后尝试打开浏览器
ui.launch(inbrowser=True)
